# Notebook 02 — Dimensionality Assessment

**Goal**: Prove essential unidimensionality of the Russian ART using 5 complementary methods, **real-author items only** (no foils).

Methods:
- 2a. Exploratory Factor Analysis (EFA) on tetrachoric correlations
- 2b. Confirmatory Factor Analysis (CFA) — one-factor model
- 2c. IRT Residual Diagnostics (PCA of residuals, Q3, LD chi-square)
- 2d. UIRT vs MIRT Model Comparison
- 2e. Robustness Checks (Mokken scalability)

In [1]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.linalg import eigh
from sklearn.decomposition import PCA
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
import factor_analyzer.factor_analyzer as fa_mod
from sklearn.utils.validation import check_array as sk_check_array

# Patch factor_analyzer for newer scikit-learn
def _patched_check_array(array, *args, **kwargs):
    if "force_all_finite" in kwargs:
        kwargs["ensure_all_finite"] = kwargs.pop("force_all_finite")
    return sk_check_array(array, *args, **kwargs)
fa_mod.check_array = _patched_check_array

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Resolve project root
_MARKER = "data/processed/irt_pipeline/sample1_pretest.csv"
PROJECT_ROOT = os.path.abspath(os.getcwd())
while PROJECT_ROOT:
    if os.path.isfile(os.path.join(PROJECT_ROOT, _MARKER)):
        break
    _parent = os.path.dirname(PROJECT_ROOT)
    if _parent == PROJECT_ROOT:
        PROJECT_ROOT = os.path.abspath(os.getcwd())
        break
    PROJECT_ROOT = _parent

DATA_DIR = os.path.join(PROJECT_ROOT, "data/processed/irt_pipeline")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "data/processed/results/02_dimensionality")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load author-only response matrices
s1_auth = pd.read_csv(os.path.join(DATA_DIR, "sample1_authors_only.csv"))
s2_auth = pd.read_csv(os.path.join(DATA_DIR, "sample2_authors_only.csv"))
meta = pd.read_csv(os.path.join(DATA_DIR, "item_metadata.csv"))

# Coerce to numeric
s1_auth = s1_auth.apply(pd.to_numeric, errors='coerce')
s2_auth = s2_auth.apply(pd.to_numeric, errors='coerce')

author_cols = s1_auth.columns.tolist()
print(f"Sample 1: {s1_auth.shape}, Sample 2: {s2_auth.shape}")
print(f"Author items: {len(author_cols)}")

Sample 1: (1035, 101), Sample 2: (800, 101)
Author items: 101


---
## 2a. Exploratory Factor Analysis (EFA)
### Tetrachoric correlation matrix

In [2]:
def compute_tetrachoric(data):
    """Compute tetrachoric correlation matrix for binary items.
    Uses the cos(pi / (1 + sqrt(ad/bc))) approximation (Bonett & Price, 2005).
    """
    n_items = data.shape[1]
    corr = np.eye(n_items)
    cols = data.columns
    
    for i in range(n_items):
        for j in range(i+1, n_items):
            x = data.iloc[:, i].dropna()
            y = data.iloc[:, j].dropna()
            common = x.index.intersection(y.index)
            x_c = x.loc[common].values
            y_c = y.loc[common].values
            
            a = np.sum((x_c == 1) & (y_c == 1))
            b = np.sum((x_c == 1) & (y_c == 0))
            c = np.sum((x_c == 0) & (y_c == 1))
            d = np.sum((x_c == 0) & (y_c == 0))
            
            # Handle edge cases
            if a*d == 0 and b*c == 0:
                r_tet = 0.0
            elif b*c == 0:
                r_tet = 1.0
            elif a*d == 0:
                r_tet = -1.0
            else:
                odds_ratio = (a * d) / (b * c)
                r_tet = np.cos(np.pi / (1 + np.sqrt(odds_ratio)))
            
            corr[i, j] = r_tet
            corr[j, i] = r_tet
    
    return pd.DataFrame(corr, index=cols, columns=cols)

print("Computing tetrachoric correlation matrices (this takes ~1 min per sample)...")
tet1 = compute_tetrachoric(s1_auth)
print(f"Sample 1 tetrachoric matrix: {tet1.shape}")
tet2 = compute_tetrachoric(s2_auth)
print(f"Sample 2 tetrachoric matrix: {tet2.shape}")

# Verify positive semi-definiteness; if not, apply nearest PSD correction
for name, tet in [('S1', tet1), ('S2', tet2)]:
    eigvals = np.linalg.eigvalsh(tet.values)
    n_neg = (eigvals < -1e-10).sum()
    print(f"{name}: min eigenvalue = {eigvals.min():.6f}, negative eigenvalues: {n_neg}")

Computing tetrachoric correlation matrices (this takes ~1 min per sample)...


Sample 1 tetrachoric matrix: (101, 101)


Sample 2 tetrachoric matrix: (101, 101)
S1: min eigenvalue = -2.832719, negative eigenvalues: 29
S2: min eigenvalue = -5.386642, negative eigenvalues: 36


### KMO and Bartlett's test

In [3]:
def make_psd(corr_matrix, eps=1e-6):
    """Ensure a correlation matrix is positive semi-definite."""
    eigvals, eigvecs = np.linalg.eigh(corr_matrix)
    eigvals = np.maximum(eigvals, eps)
    psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    # Re-normalize to correlation matrix
    d = np.sqrt(np.diag(psd))
    psd = psd / np.outer(d, d)
    np.fill_diagonal(psd, 1.0)
    return psd

print("="*60)
print("KMO and Bartlett's Test of Sphericity")
print("="*60)

for name, data, tet in [('Sample 1', s1_auth, tet1), ('Sample 2', s2_auth, tet2)]:
    # KMO on the raw data (Pearson-based, standard approach)
    kmo_all, kmo_model = calculate_kmo(data.dropna())
    
    # Bartlett's test
    chi2_bart, p_bart = calculate_bartlett_sphericity(data.dropna())
    
    print(f"\n{name}:")
    print(f"  KMO = {kmo_model:.4f}")
    print(f"  Bartlett's chi2 = {chi2_bart:.1f}, p = {p_bart:.2e}")
    if kmo_model >= 0.80:
        print(f"  KMO interpretation: meritorious (>= 0.80)")
    elif kmo_model >= 0.70:
        print(f"  KMO interpretation: middling (>= 0.70)")
    elif kmo_model >= 0.60:
        print(f"  KMO interpretation: mediocre (>= 0.60)")
    else:
        print(f"  KMO interpretation: unacceptable (< 0.60)")

KMO and Bartlett's Test of Sphericity

Sample 1:
  KMO = 0.9424
  Bartlett's chi2 = 20875.7, p = 0.00e+00
  KMO interpretation: meritorious (>= 0.80)



Sample 2:
  KMO = 0.9458
  Bartlett's chi2 = 31115.1, p = 0.00e+00
  KMO interpretation: meritorious (>= 0.80)


/home/polina/Documents/Cursor_Projects/Russian Author Recognition Test Cursor/.venv/lib/python3.12/site-packages/factor_analyzer/utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(


### Parallel analysis and scree plot

In [4]:
def parallel_analysis_pearson(data, n_rep=100, percentile=95, seed=42):
    """Horn's parallel analysis using column-wise permutation
    with Pearson correlations (standard approach).
    """
    n, p = data.shape
    obs_corr = np.corrcoef(data.values.T)
    obs_eigvals = np.sort(np.linalg.eigvalsh(obs_corr))[::-1]

    rng = np.random.RandomState(seed)
    rand_eigvals = np.zeros((n_rep, p))

    for r in range(n_rep):
        shuffled = data.values.copy()
        for col in range(p):
            rng.shuffle(shuffled[:, col])
        rand_corr = np.corrcoef(shuffled.T)
        rand_corr = np.nan_to_num(rand_corr, nan=0.0)
        np.fill_diagonal(rand_corr, 1.0)
        try:
            rand_eigvals[r] = np.sort(np.linalg.eigvalsh(rand_corr))[::-1]
        except np.linalg.LinAlgError:
            rand_corr = make_psd(rand_corr)
            rand_eigvals[r] = np.sort(np.linalg.eigvalsh(rand_corr))[::-1]

    threshold = np.percentile(rand_eigvals, percentile, axis=0)
    n_factors = np.sum(obs_eigvals > threshold)
    return obs_eigvals, threshold, n_factors

# Tetrachoric eigenvalues (already computed above)
tet1_psd = make_psd(tet1.values)
tet2_psd = make_psd(tet2.values)
tet_eig1 = np.sort(np.linalg.eigvalsh(tet1_psd))[::-1]
tet_eig2 = np.sort(np.linalg.eigvalsh(tet2_psd))[::-1]
print("Tetrachoric eigenvalues (first 5):")
print(f"  Sample 1: {tet_eig1[:5].round(3)}")
print(f"  Sample 2: {tet_eig2[:5].round(3)}")

print("\nRunning parallel analysis (Pearson, Sample 1)...")
eig1, thr1, nf1 = parallel_analysis_pearson(s1_auth.dropna())
print(f"  Suggested factors: {nf1}")

print("Running parallel analysis (Pearson, Sample 2)...")
eig2, thr2, nf2 = parallel_analysis_pearson(s2_auth.dropna())
print(f"  Suggested factors: {nf2}")

Tetrachoric eigenvalues (first 5):
  Sample 1: [44.486  6.184  4.506  4.205  3.28 ]
  Sample 2: [33.292 10.578  4.497  4.179  3.26 ]

Running parallel analysis (Pearson, Sample 1)...


  Suggested factors: 6
Running parallel analysis (Pearson, Sample 2)...


  Suggested factors: 8


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
n_show = 15

for ax, eig, thr, nf, name in [
    (axes[0], eig1, thr1, nf1, 'Sample 1'),
    (axes[1], eig2, thr2, nf2, 'Sample 2')
]:
    x = range(1, n_show + 1)
    ax.plot(x, eig[:n_show], 'bo-', label='Observed eigenvalues')
    ax.plot(x, thr[:n_show], 'r--', label='95th percentile (random)')
    ax.set_xlabel('Factor Number')
    ax.set_ylabel('Eigenvalue')
    ax.set_title(f'{name}: Scree Plot with Parallel Analysis\n(Suggested factors: {nf})')
    ax.legend()
    ax.set_xticks(list(x))

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'scree_parallel_analysis.png'), dpi=150)
plt.show()

# Eigenvalue metrics
for name, eig, t_eig in [('Sample 1', eig1, tet_eig1), ('Sample 2', eig2, tet_eig2)]:
    ratio = t_eig[0] / t_eig[1]
    var1 = 100 * t_eig[0] / t_eig.sum()
    print(f"\n{name}:")
    print(f"  1st eigenvalue (tetrachoric): {t_eig[0]:.3f}")
    print(f"  2nd eigenvalue (tetrachoric): {t_eig[1]:.3f}")
    print(f"  Ratio (1st/2nd): {ratio:.2f} {'(> 3.0 = strong)' if ratio > 3 else '(< 3.0)'}")
    print(f"  Variance by 1st factor: {var1:.1f}% {'(> 20% = adequate)' if var1 > 20 else '(< 20%)'}")


Sample 1:
  1st eigenvalue (tetrachoric): 44.486
  2nd eigenvalue (tetrachoric): 6.184
  Ratio (1st/2nd): 7.19 (> 3.0 = strong)
  Variance by 1st factor: 44.0% (> 20% = adequate)

Sample 2:
  1st eigenvalue (tetrachoric): 33.292
  2nd eigenvalue (tetrachoric): 10.578
  Ratio (1st/2nd): 3.15 (> 3.0 = strong)
  Variance by 1st factor: 33.0% (> 20% = adequate)


/tmp/ipykernel_43258/1830278586.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### EFA: 1-factor and 2-factor models

In [6]:
def run_efa(data, n_factors, name):
    """Run EFA on data with given number of factors."""
    data_clean = data.dropna()
    rotation = 'promax' if n_factors > 1 else None
    
    fa = FactorAnalyzer(n_factors=n_factors, rotation=rotation, method='minres')
    fa.fit(data_clean)
    
    loadings = pd.DataFrame(
        fa.loadings_,
        index=data_clean.columns,
        columns=[f'Factor{i+1}' for i in range(n_factors)]
    )
    communalities = pd.Series(fa.get_communalities(), index=data_clean.columns)
    eigenvalues = fa.get_eigenvalues()
    
    print(f"\n{name} — {n_factors}-factor EFA:")
    print(f"  Rotation: {rotation}")
    
    if n_factors == 1:
        low_loading = (loadings['Factor1'].abs() < 0.30).sum()
        print(f"  Mean |loading|: {loadings['Factor1'].abs().mean():.3f}")
        print(f"  Items with |loading| < 0.30: {low_loading}")
        print(f"  Mean communality: {communalities.mean():.3f}")
    else:
        for f in loadings.columns:
            print(f"  {f}: mean |loading| = {loadings[f].abs().mean():.3f}")
    
    return loadings, communalities, eigenvalues

print("="*60)
print("Exploratory Factor Analysis")
print("="*60)

# Sample 1
load1_1f, comm1_1f, eig1_1f = run_efa(s1_auth, 1, 'Sample 1')
load1_2f, comm1_2f, eig1_2f = run_efa(s1_auth, 2, 'Sample 1')

# Sample 2
load2_1f, comm2_1f, eig2_1f = run_efa(s2_auth, 1, 'Sample 2')
load2_2f, comm2_2f, eig2_2f = run_efa(s2_auth, 2, 'Sample 2')

# Save loadings
load1_1f.to_csv(os.path.join(RESULTS_DIR, 'efa_loadings_1f_sample1.csv'))
load2_1f.to_csv(os.path.join(RESULTS_DIR, 'efa_loadings_1f_sample2.csv'))

Exploratory Factor Analysis



Sample 1 — 1-factor EFA:
  Rotation: None
  Mean |loading|: 0.459
  Items with |loading| < 0.30: 24
  Mean communality: 0.240



Sample 1 — 2-factor EFA:
  Rotation: promax
  Factor1: mean |loading| = 0.343
  Factor2: mean |loading| = 0.249



Sample 2 — 1-factor EFA:
  Rotation: None
  Mean |loading|: 0.381
  Items with |loading| < 0.30: 37
  Mean communality: 0.195



Sample 2 — 2-factor EFA:
  Rotation: promax
  Factor1: mean |loading| = 0.376
  Factor2: mean |loading| = 0.156


---
## 2b. Confirmatory Factor Analysis (CFA)

One-factor CFA on binary items using `semopy`.

In [7]:
import semopy

def run_cfa_one_factor(data, name):
    """Run one-factor CFA using semopy."""
    data_clean = data.dropna().copy()
    
    # Sanitize column names for semopy formula
    safe_names = {}
    for i, col in enumerate(data_clean.columns):
        safe_name = f'item_{i:03d}'
        safe_names[col] = safe_name
    data_safe = data_clean.rename(columns=safe_names)
    
    # Build model specification
    items_str = ' + '.join(data_safe.columns)
    model_spec = f'F1 =~ {items_str}'
    
    model = semopy.Model(model_spec)
    result = model.fit(data_safe)
    
    # Extract fit statistics
    stats_df = semopy.calc_stats(model)
    
    print(f"\n{'='*60}")
    print(f"CFA One-Factor Model: {name}")
    print(f"{'='*60}")
    
    fit_metrics = {}
    for col in stats_df.columns:
        val = stats_df[col].values[0]
        fit_metrics[col] = val
        print(f"  {col}: {val}")
    
    # Factor loadings
    estimates = model.inspect()
    loadings = estimates[estimates['op'] == '~']
    
    # Map safe names back
    reverse_map = {v: k for k, v in safe_names.items()}
    loading_vals = []
    for _, row in loadings.iterrows():
        orig_name = reverse_map.get(row['rval'], row['rval'])
        loading_vals.append({'item': orig_name, 'loading': row['Estimate']})
    
    loading_df = pd.DataFrame(loading_vals)
    if len(loading_df) > 0:
        low_load = (loading_df['loading'].abs() < 0.40).sum()
        print(f"\n  Factor loadings:")
        print(f"    Mean: {loading_df['loading'].mean():.3f}")
        print(f"    SD:   {loading_df['loading'].std():.3f}")
        print(f"    Items with |loading| < 0.40: {low_load}")
    
    # Interpret fit indices
    print(f"\n  Fit interpretation:")
    if 'RMSEA' in fit_metrics:
        rmsea = fit_metrics['RMSEA']
        if rmsea < 0.06:
            print(f"    RMSEA = {rmsea:.4f}: GOOD (< 0.06)")
        elif rmsea < 0.08:
            print(f"    RMSEA = {rmsea:.4f}: ACCEPTABLE (< 0.08)")
        else:
            print(f"    RMSEA = {rmsea:.4f}: POOR (>= 0.08)")
    
    if 'CFI' in fit_metrics:
        cfi = fit_metrics['CFI']
        if cfi > 0.95:
            print(f"    CFI = {cfi:.4f}: GOOD (> 0.95)")
        elif cfi > 0.90:
            print(f"    CFI = {cfi:.4f}: ACCEPTABLE (> 0.90)")
        else:
            print(f"    CFI = {cfi:.4f}: POOR (<= 0.90)")
    
    if 'TLI' in fit_metrics:
        tli = fit_metrics['TLI']
        if tli > 0.95:
            print(f"    TLI = {tli:.4f}: GOOD (> 0.95)")
        elif tli > 0.90:
            print(f"    TLI = {tli:.4f}: ACCEPTABLE (> 0.90)")
        else:
            print(f"    TLI = {tli:.4f}: POOR (<= 0.90)")
    
    return fit_metrics, loading_df

cfa1_fit, cfa1_load = run_cfa_one_factor(s1_auth, 'Sample 1')
cfa2_fit, cfa2_load = run_cfa_one_factor(s2_auth, 'Sample 2')


CFA One-Factor Model: Sample 1
  DoF: 4949
  DoF Baseline: 5050
  chi2: 10913.129574558872
  chi2 p-value: 0.0
  chi2 Baseline: 22667.95633123019
  CFI: 0.6614743808856733
  GFI: 0.5185657932681126
  AGFI: 0.5087406053756252
  NFI: 0.5185657932681126
  TLI: 0.6545656947812993
  RMSEA: 0.05186516945866042
  AIC: 355.38917784160856
  BIC: 1185.0078011655442
  LogLik: 24.305411079195707



  Factor loadings:
    Mean: 1.942
    SD:   1.023
    Items with |loading| < 0.40: 12

  Fit interpretation:
    RMSEA = 0.0519: GOOD (< 0.06)
    CFI = 0.6615: POOR (<= 0.90)
    TLI = 0.6546: POOR (<= 0.90)



CFA One-Factor Model: Sample 2
  DoF: 4949
  DoF Baseline: 5050
  chi2: 16057.431312842822
  chi2 p-value: 0.0
  chi2 Baseline: 32563.720273256025
  CFI: 0.5962584774971171
  GFI: 0.5068919896713862
  AGFI: 0.4968285608891695
  NFI: 0.5068919896713863
  TLI: 0.588018854588895
  RMSEA: 0.05306870370793582
  AIC: 363.75581124600797
  BIC: 1309.5417479308694
  LogLik: 20.122094376996017



  Factor loadings:
    Mean: 1.189
    SD:   0.880
    Items with |loading| < 0.40: 32

  Fit interpretation:
    RMSEA = 0.0531: GOOD (< 0.06)
    CFI = 0.5963: POOR (<= 0.90)
    TLI = 0.5880: POOR (<= 0.90)


---
## 2c. IRT Residual Diagnostics

Fit 2PL model, compute standardized residuals, PCA of residuals, Q3 statistic, LD chi-square.

In [8]:
import girth

def fit_2pl_and_residuals(data, name):
    """Fit 2PL, compute expected probabilities, and residuals."""
    data_clean = data.dropna().copy()
    X = data_clean.values.astype(int)
    
    # Fit 2PL (girth expects items x persons, integer 0/1)
    params = girth.twopl_mml(X.T)
    a_params = params['Discrimination']
    b_params = params['Difficulty']
    
    # EAP ability estimates
    theta = girth.ability_eap(X.T, b_params, a_params)
    
    # Expected probabilities P(X=1 | theta, a, b)
    # P = 1 / (1 + exp(-a*(theta - b)))
    n_persons, n_items = X.shape
    P = np.zeros_like(X, dtype=float)
    for j in range(n_items):
        P[:, j] = 1.0 / (1.0 + np.exp(-a_params[j] * (theta - b_params[j])))
    
    # Standardized residuals
    residuals = (X - P) / np.sqrt(P * (1 - P) + 1e-10)
    
    print(f"\n{name} — 2PL fit:")
    print(f"  Items: {n_items}, Persons: {n_persons}")
    print(f"  a: M = {a_params.mean():.3f}, SD = {a_params.std():.3f}")
    print(f"  b: M = {b_params.mean():.3f}, SD = {b_params.std():.3f}")
    
    return a_params, b_params, theta, P, residuals, data_clean

a1, b1, theta1, P1, resid1, d1_clean = fit_2pl_and_residuals(s1_auth, 'Sample 1')
a2, b2, theta2, P2, resid2, d2_clean = fit_2pl_and_residuals(s2_auth, 'Sample 2')


Sample 1 — 2PL fit:
  Items: 101, Persons: 449
  a: M = 1.681, SD = 0.718
  b: M = 0.064, SD = 1.748



Sample 2 — 2PL fit:
  Items: 101, Persons: 798
  a: M = 1.544, SD = 0.812
  b: M = 1.231, SD = 2.816


### PCA of standardized residuals

In [9]:
def pca_of_residuals(residuals, name, n_components=10):
    """PCA on standardized residuals to check for secondary dimensions."""
    # Replace any inf/nan
    resid_clean = np.nan_to_num(residuals, nan=0.0, posinf=0.0, neginf=0.0)
    
    pca = PCA(n_components=n_components)
    pca.fit(resid_clean)
    
    explained_var = pca.explained_variance_
    
    print(f"\n{'='*60}")
    print(f"PCA of Standardized Residuals: {name}")
    print(f"{'='*60}")
    print(f"  First 5 eigenvalues: {explained_var[:5].round(3)}")
    print(f"  1st residual eigenvalue: {explained_var[0]:.3f}")
    
    if explained_var[0] < 2.0:
        print(f"  PASS: < 2.0 (essential unidimensionality supported)")
    else:
        print(f"  WARNING: >= 2.0 (possible secondary dimension)")
    
    return explained_var

resid_eig1 = pca_of_residuals(resid1, 'Sample 1')
resid_eig2 = pca_of_residuals(resid2, 'Sample 2')


PCA of Standardized Residuals: Sample 1
  First 5 eigenvalues: [9.595 4.83  4.342 3.887 3.241]
  1st residual eigenvalue: 9.595



PCA of Standardized Residuals: Sample 2
  First 5 eigenvalues: [9.568 5.499 3.897 3.886 3.027]
  1st residual eigenvalue: 9.568


### Q3 statistic (Yen, 1984) and Local Dependence

In [10]:
def q3_and_ld(residuals, item_names, name, ld_threshold=10.0, q3_threshold=0.20):
    """Compute Q3 (residual correlations) and LD chi-square statistics."""
    resid_clean = np.nan_to_num(residuals, nan=0.0, posinf=0.0, neginf=0.0)
    n_items = resid_clean.shape[1]
    n_persons = resid_clean.shape[0]
    
    # Q3 = correlation of residuals between item pairs
    q3_matrix = np.corrcoef(resid_clean.T)
    np.fill_diagonal(q3_matrix, 0)  # ignore self-correlations
    
    # LD chi-square: n * r^2 for each pair
    ld_matrix = n_persons * q3_matrix ** 2
    
    # Count flagged pairs
    n_pairs = n_items * (n_items - 1) // 2
    upper_q3 = q3_matrix[np.triu_indices(n_items, k=1)]
    upper_ld = ld_matrix[np.triu_indices(n_items, k=1)]
    
    q3_flagged = np.sum(np.abs(upper_q3) > q3_threshold)
    ld_flagged = np.sum(upper_ld > ld_threshold)
    
    # Bonferroni-corrected p-values for LD
    ld_pvals = 1 - stats.chi2.cdf(upper_ld, df=1)
    bonferroni_threshold = 0.05 / n_pairs
    ld_sig_bonf = np.sum(ld_pvals < bonferroni_threshold)
    
    print(f"\n{'='*60}")
    print(f"Q3 and Local Dependence: {name}")
    print(f"{'='*60}")
    print(f"  Total item pairs: {n_pairs}")
    print(f"  Q3 statistics:")
    print(f"    Mean |Q3|: {np.abs(upper_q3).mean():.4f}")
    print(f"    Max |Q3|:  {np.abs(upper_q3).max():.4f}")
    print(f"    Pairs with |Q3| > {q3_threshold}: {q3_flagged} ({100*q3_flagged/n_pairs:.1f}%)")
    print(f"  LD chi-square:")
    print(f"    Pairs with LD > {ld_threshold}: {ld_flagged} ({100*ld_flagged/n_pairs:.1f}%)")
    print(f"    Significant after Bonferroni (p < {bonferroni_threshold:.2e}): {ld_sig_bonf}")
    
    return q3_matrix, ld_matrix

q3_1, ld_1 = q3_and_ld(resid1, author_cols, 'Sample 1')
q3_2, ld_2 = q3_and_ld(resid2, author_cols, 'Sample 2')


Q3 and Local Dependence: Sample 1
  Total item pairs: 5050
  Q3 statistics:
    Mean |Q3|: 0.0576
    Max |Q3|:  0.6517
    Pairs with |Q3| > 0.2: 58 (1.1%)
  LD chi-square:
    Pairs with LD > 10.0: 257 (5.1%)
    Significant after Bonferroni (p < 9.90e-06): 51



Q3 and Local Dependence: Sample 2
  Total item pairs: 5050
  Q3 statistics:
    Mean |Q3|: 0.0524
    Max |Q3|:  0.4888
    Pairs with |Q3| > 0.2: 83 (1.6%)
  LD chi-square:
    Pairs with LD > 10.0: 526 (10.4%)
    Significant after Bonferroni (p < 9.90e-06): 203


---
## 2d. UIRT vs MIRT Model Comparison

Compare 1-factor vs 2-factor models using EFA goodness-of-fit and information criteria.

In [11]:
def compare_factor_models(data, name):
    """Compare 1-factor vs 2-factor models."""
    data_clean = data.dropna()
    n = len(data_clean)
    p = data_clean.shape[1]
    
    # Fit 1-factor model
    fa1 = FactorAnalyzer(n_factors=1, rotation=None, method='ml')
    try:
        fa1.fit(data_clean)
        # Use eigenvalues as proxy for variance explained
        ev1 = fa1.get_eigenvalues()
        # Log-likelihood approximation via chi-square test
        # factor_analyzer provides a Bartlett's chi-square for the model
        chi2_1, df_1 = None, None
    except Exception:
        pass
    
    # Fit 2-factor model
    fa2 = FactorAnalyzer(n_factors=2, rotation='promax', method='ml')
    try:
        fa2.fit(data_clean)
    except Exception:
        pass
    
    # Compare using cumulative variance explained
    var_1f = fa1.get_factor_variance()
    var_2f = fa2.get_factor_variance()
    
    print(f"\n{'='*60}")
    print(f"Factor Model Comparison: {name}")
    print(f"{'='*60}")
    
    cum_var_1f = var_1f[2][0]  # Cumulative variance for 1 factor
    cum_var_2f = var_2f[2][-1]  # Cumulative variance for 2 factors
    
    print(f"  1-factor: cumulative variance = {cum_var_1f:.4f}")
    print(f"  2-factor: cumulative variance = {cum_var_2f:.4f}")
    print(f"  Increment from 2nd factor: {cum_var_2f - cum_var_1f:.4f}")
    
    # Additional: BIC-like comparison using eigenvalues
    eig_obs = fa1.get_eigenvalues()[0]  # Original eigenvalues
    ratio_12 = eig_obs[0] / eig_obs[1] if eig_obs[1] > 0 else float('inf')
    ratio_23 = eig_obs[1] / eig_obs[2] if eig_obs[2] > 0 else float('inf')
    
    print(f"\n  Eigenvalue ratios:")
    print(f"    1st/2nd: {ratio_12:.2f}")
    print(f"    2nd/3rd: {ratio_23:.2f}")
    
    if ratio_12 > 3.0 and (cum_var_2f - cum_var_1f) < 0.05:
        print(f"  CONCLUSION: Unidimensional model is adequate")
    else:
        print(f"  CONCLUSION: Some evidence of multidimensionality; "
              f"inspect factor loadings")
    
    return var_1f, var_2f

var1_1f, var1_2f = compare_factor_models(s1_auth, 'Sample 1')
var2_1f, var2_2f = compare_factor_models(s2_auth, 'Sample 2')


Factor Model Comparison: Sample 1
  1-factor: cumulative variance = 0.2395
  2-factor: cumulative variance = 0.2505
  Increment from 2nd factor: 0.0111

  Eigenvalue ratios:
    1st/2nd: 5.64
    2nd/3rd: 1.11
  CONCLUSION: Unidimensional model is adequate



Factor Model Comparison: Sample 2
  1-factor: cumulative variance = 0.1949
  2-factor: cumulative variance = 0.2368
  Increment from 2nd factor: 0.0420



  Eigenvalue ratios:
    1st/2nd: 3.98
    2nd/3rd: 1.38
  CONCLUSION: Unidimensional model is adequate


---
## 2e. Robustness Check: Mokken Scalability Analysis

In [12]:
def mokken_h_coefficient(data, name):
    """Compute Loevinger's H coefficients for Mokken scale analysis.
    H = 1 - (observed Guttman errors) / (expected Guttman errors under independence)
    """
    data_clean = data.dropna().values.astype(float)
    n, k = data_clean.shape
    
    # Item means (difficulty)
    p = data_clean.mean(axis=0)
    
    # Pairwise H_ij
    H_ij = np.zeros((k, k))
    total_obs_err = 0
    total_exp_err = 0
    
    for i in range(k):
        for j in range(i+1, k):
            # Make sure p_i >= p_j (easier item first)
            if p[i] < p[j]:
                xi, xj = data_clean[:, j], data_clean[:, i]
                pi, pj = p[j], p[i]
            else:
                xi, xj = data_clean[:, i], data_clean[:, j]
                pi, pj = p[i], p[j]
            
            # Guttman error: person scores 0 on easy item but 1 on hard item
            obs_err = np.sum((xi == 0) & (xj == 1))
            # Expected errors under independence
            exp_err = n * (1 - pi) * pj
            
            if exp_err > 0:
                H_ij[i, j] = 1 - obs_err / exp_err
            else:
                H_ij[i, j] = 1.0
            H_ij[j, i] = H_ij[i, j]
            
            total_obs_err += obs_err
            total_exp_err += exp_err
    
    # Overall H
    H_total = 1 - total_obs_err / total_exp_err if total_exp_err > 0 else 0
    
    # Per-item H_i
    H_i = np.zeros(k)
    for i in range(k):
        obs_i = 0
        exp_i = 0
        for j in range(k):
            if i == j:
                continue
            if p[i] < p[j]:
                xi, xj = data_clean[:, j], data_clean[:, i]
                pi, pj = p[j], p[i]
            else:
                xi, xj = data_clean[:, i], data_clean[:, j]
                pi, pj = p[i], p[j]
            obs_i += np.sum((xi == 0) & (xj == 1))
            exp_i += n * (1 - pi) * pj
        H_i[i] = 1 - obs_i / exp_i if exp_i > 0 else 0
    
    # Monotonicity check: for each item, rest-score should be non-decreasing
    cols = data.dropna().columns
    violations = 0
    for i in range(k):
        rest_score = data_clean[:, [j for j in range(k) if j != i]].sum(axis=1)
        unique_rest = np.unique(rest_score)
        prev_prop = -1
        for rs in sorted(unique_rest):
            mask = rest_score == rs
            if mask.sum() < 5:
                continue
            prop = data_clean[mask, i].mean()
            if prop < prev_prop - 0.01:  # small tolerance
                violations += 1
                break
            prev_prop = prop
    
    print(f"\n{'='*60}")
    print(f"Mokken Scalability Analysis: {name}")
    print(f"{'='*60}")
    print(f"  Overall H = {H_total:.4f}")
    if H_total >= 0.50:
        print(f"  Interpretation: STRONG scale (H >= 0.50)")
    elif H_total >= 0.40:
        print(f"  Interpretation: MEDIUM scale (H >= 0.40)")
    elif H_total >= 0.30:
        print(f"  Interpretation: WEAK scale (H >= 0.30)")
    else:
        print(f"  Interpretation: Does NOT form a Mokken scale (H < 0.30)")
    
    print(f"\n  Per-item H:")
    print(f"    Mean: {H_i.mean():.4f}, SD: {H_i.std():.4f}")
    print(f"    Min:  {H_i.min():.4f} (item: {cols[np.argmin(H_i)]})")
    print(f"    Max:  {H_i.max():.4f} (item: {cols[np.argmax(H_i)]})")
    n_below_030 = (H_i < 0.30).sum()
    print(f"    Items with H_i < 0.30: {n_below_030}")
    
    print(f"\n  Monotonicity check:")
    print(f"    Items with monotonicity violations: {violations} / {k}")
    
    return H_total, H_i

H1_total, H1_items = mokken_h_coefficient(s1_auth, 'Sample 1')
H2_total, H2_items = mokken_h_coefficient(s2_auth, 'Sample 2')


Mokken Scalability Analysis: Sample 1
  Overall H = 0.4137
  Interpretation: MEDIUM scale (H >= 0.40)

  Per-item H:
    Mean: 0.4243, SD: 0.1013
    Min:  0.1516 (item: Paula Hawkins)
    Max:  0.6934 (item: Charles Dickens)
    Items with H_i < 0.30: 10

  Monotonicity check:
    Items with monotonicity violations: 101 / 101



Mokken Scalability Analysis: Sample 2
  Overall H = 0.4024
  Interpretation: MEDIUM scale (H >= 0.40)

  Per-item H:
    Mean: 0.3715, SD: 0.1507
    Min:  0.0823 (item: Alan Milne)
    Max:  0.6772 (item: Charles Dickens)
    Items with H_i < 0.30: 32

  Monotonicity check:
    Items with monotonicity violations: 101 / 101


---
## Summary Table

In [13]:
def safe_get(d, key, default='N/A'):
    v = d.get(key, default)
    if isinstance(v, float):
        return f"{v:.4f}"
    return str(v)

summary = pd.DataFrame({
    'Metric': [
        '1st eigenvalue (tetrachoric)',
        '2nd eigenvalue (tetrachoric)',
        'Eigenvalue ratio (1st/2nd)',
        'Variance by 1st factor (%)',
        'Parallel analysis: suggested factors',
        'KMO',
        'CFA RMSEA',
        'CFA CFI',
        'CFA TLI',
        'Residual PCA: 1st eigenvalue',
        'Q3: pairs |Q3| > 0.20',
        'Mokken H (overall)',
    ],
    'Sample 1': [
        f"{tet_eig1[0]:.3f}",
        f"{tet_eig1[1]:.3f}",
        f"{tet_eig1[0]/tet_eig1[1]:.2f}",
        f"{100*tet_eig1[0]/tet_eig1.sum():.1f}",
        str(nf1),
        safe_get(cfa1_fit, 'KMO', 'see above'),
        safe_get(cfa1_fit, 'RMSEA'),
        safe_get(cfa1_fit, 'CFI'),
        safe_get(cfa1_fit, 'TLI'),
        f"{resid_eig1[0]:.3f}",
        f"{np.sum(np.abs(q3_1[np.triu_indices(len(author_cols), k=1)]) > 0.20)}",
        f"{H1_total:.4f}",
    ],
    'Sample 2': [
        f"{tet_eig2[0]:.3f}",
        f"{tet_eig2[1]:.3f}",
        f"{tet_eig2[0]/tet_eig2[1]:.2f}",
        f"{100*tet_eig2[0]/tet_eig2.sum():.1f}",
        str(nf2),
        safe_get(cfa2_fit, 'KMO', 'see above'),
        safe_get(cfa2_fit, 'RMSEA'),
        safe_get(cfa2_fit, 'CFI'),
        safe_get(cfa2_fit, 'TLI'),
        f"{resid_eig2[0]:.3f}",
        f"{np.sum(np.abs(q3_2[np.triu_indices(len(author_cols), k=1)]) > 0.20)}",
        f"{H2_total:.4f}",
    ],
    'Criterion': [
        'Large (dominant)',
        'Much smaller than 1st',
        '> 3.0',
        '> 20%',
        '1',
        '>= 0.60',
        '< 0.08',
        '> 0.90',
        '> 0.90',
        '< 2.0',
        'Few/none',
        '>= 0.30',
    ]
})

print("\n" + "="*60)
print("DIMENSIONALITY ASSESSMENT SUMMARY")
print("="*60)
print(summary.to_string(index=False))

summary.to_csv(os.path.join(RESULTS_DIR, 'dimensionality_summary.csv'), index=False)
print(f"\nSaved to: {RESULTS_DIR}")


DIMENSIONALITY ASSESSMENT SUMMARY
                              Metric  Sample 1  Sample 2             Criterion
        1st eigenvalue (tetrachoric)    44.486    33.292      Large (dominant)
        2nd eigenvalue (tetrachoric)     6.184    10.578 Much smaller than 1st
          Eigenvalue ratio (1st/2nd)      7.19      3.15                 > 3.0
          Variance by 1st factor (%)      44.0      33.0                 > 20%
Parallel analysis: suggested factors         6         8                     1
                                 KMO see above see above               >= 0.60
                           CFA RMSEA    0.0519    0.0531                < 0.08
                             CFA CFI    0.6615    0.5963                > 0.90
                             CFA TLI    0.6546    0.5880                > 0.90
        Residual PCA: 1st eigenvalue     9.595     9.568                 < 2.0
               Q3: pairs |Q3| > 0.20        58        83              Few/none
                 